# Precompute Graphs for Station Network

This notebook builds three graph types from your prepared dataset and `Graph_build.py`:

1. Static geographic adjacency using station coordinates.
2. Wind/cloud-informed adjacency using `wind_sp`, `wind_dir`, and `tcc`.
3. Time-varying DTW adjacency graphs for each timestep using a user-defined window length `L`.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def find_repo_root(start_dirs):
    for start in start_dirs:
        start = Path(start).expanduser().resolve()

        for candidate in [start, *start.parents]:
            if (candidate / "Src").exists() and (candidate / "NoteBooks").exists():
                return candidate

        for child in start.rglob("Src"):
            if child.is_dir():
                parent = child.parent
                if (parent / "NoteBooks").exists() or (parent / "README.md").exists():
                    return parent

    return None


candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/GHI"),
    Path("/content/drive/MyDrive/GHI"),
    Path("/workspace/GHI"),
    Path("/home/jovyan/work/GHI"),
]

# If the notebook is running from a GitHub clone, this is usually enough.
if "google.colab" in sys.modules:
    candidates.extend([
        Path("/content"),
        Path("/content/drive/MyDrive"),
    ])

repo_root = find_repo_root(candidates)
if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the repository root containing the 'Src' package. "
        "Clone the GitHub repo into the current runtime or mount the folder that contains it."
    )

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

base_dir = repo_root

from Src.Graph_build import (
    build_geo_matrices,
    build_static_adjacency,
    build_wind_cloud_adjacency,
    build_dtw_graphs_from_timeseries,
)

filtered_dir = os.path.join(base_dir, "CYL_GHI", "filtered_files")
geo_path = os.path.join(base_dir, "CYL_geo", "CyL_geo.csv")
tensor_path = os.path.join(filtered_dir, "filtered_tensor.npz")

print("repo_root:", repo_root)
print("filtered_dir:", filtered_dir)
print("geo_path:", geo_path)
print("tensor_path:", tensor_path)

In [ ]:
# Load station geometry and build the static adjacency graph
if not os.path.exists(geo_path):
    raise FileNotFoundError(f'Geo CSV not found: {geo_path}')

geo_df = pd.read_csv(geo_path)
geo_mats = build_geo_matrices(geo_df, device='cpu')
static_graph = build_static_adjacency(
    dist_matrix=geo_mats['dist_matrix'],
    k=8,
    self_loops=False,
    topk_sym=True,
)

print('Static graph keys:', list(static_graph.keys()))
print('Static adjacency shape:', static_graph['A_sym_norm'].shape)

In [ ]:
# Load precomputed tensor data and align features for wind/cloud graph building
if not os.path.exists(tensor_path):
    raise FileNotFoundError(f'Tensor file not found: {tensor_path}')

with np.load(tensor_path, allow_pickle=True) as data:
    tensor = data['tensor']
    features = data['features'].astype(str)

print('tensor shape:', tensor.shape)
print('features:', list(features))

# Select the features used for wind/cloud graphs
if 'wind_sp' in features and 'wind_dir' in features and 'tcc' in features:
    wind_sp = torch.from_numpy(tensor[:, :, features == 'wind_sp'].squeeze(-1).astype(np.float32))
    wind_dir = torch.from_numpy(tensor[:, :, features == 'wind_dir'].squeeze(-1).astype(np.float32))
    tcc = torch.from_numpy(tensor[:, :, features == 'tcc'].squeeze(-1).astype(np.float32))
else:
    raise ValueError('Required features wind_sp, wind_dir, and tcc not found in tensor.')

# Build wind/cloud adjacency for each timestep
wind_cloud_graphs = build_wind_cloud_adjacency(
    D_ij=geo_mats['dist_matrix'],
    Theta_ij=geo_mats['theta_matrix'],
    wind_sp=wind_sp,
    wind_dir=wind_dir,
    tcc=tcc,
    k=8,
    self_loops=False,
    sparse=False,
    topk_sym=True,
)

print('Wind/cloud graph keys:', list(wind_cloud_graphs.keys()))
print('Wind/cloud graph shape:', wind_cloud_graphs['A_sym_norm'].shape)

In [ ]:
# Build DTW graphs for each timestep from a user-defined window length L
selected_features = [
    'GHI_norm',
    'tcc',
    'wind_sp_norm',
    'wind_dir_sin',
    'wind_dir_cos',
]
feature_mask = np.isin(features, selected_features)
if not feature_mask.any():
    raise ValueError(f'None of the selected features were found: {selected_features}')

X = torch.from_numpy(tensor[:, :, feature_mask].astype(np.float32))
print('DTW timeseries tensor shape [T,N,F]:', X.shape)

L = 12  # user-defined window length
dtw_graphs = build_dtw_graphs_from_timeseries(
    X=X,
    L=L,
    k=8,
    self_loops=False,
    topk_sym=True,
)

print('DTW graph keys:', list(dtw_graphs.keys()))
print('DTW graph output shape:', dtw_graphs['A_sym_norm'].shape)

In [ ]:
# Save precomputed graph outputs for later use
output_dir = os.path.join(filtered_dir, 'precomputed_graphs')
os.makedirs(output_dir, exist_ok=True)

np.savez_compressed(
    os.path.join(output_dir, 'static_graph.npz'),
    A_raw=static_graph['A_raw'].numpy(),
    A_sym_norm=static_graph['A_sym_norm'].numpy(),
)
np.savez_compressed(
    os.path.join(output_dir, 'wind_cloud_graphs.npz'),
    A_sym_norm=wind_cloud_graphs['A_sym_norm'].numpy(),
    A_topk=wind_cloud_graphs['A_topk'].numpy(),
)
np.savez_compressed(
    os.path.join(output_dir, 'dtw_graphs.npz'),
    A_sym_norm=dtw_graphs['A_sym_norm'].numpy(),
    A_topk=dtw_graphs['A_topk'].numpy(),
)

print('Saved precomputed graph files to', output_dir)